# S1 SM inversion

Same De Zan inversion as [`03a_estimate_sm_nisar.ipynb`](03a_estimate_sm_nisar.ipynb)
(`calc_SM_coherences`/`invert_sm` from `INSAR4SM-main`, in-situ-driven
dry-date selection instead of ERA5 -- see that notebook's intro for why) --
but reading `S1_coherence_closure_alaska/`'s CSVs (from
[`02c_calc_s1_coherence_closure.ipynb`](02c_calc_s1_coherence_closure.ipynb))
instead of NISAR's GSLC-derived ones. Runs **SoilGrids texture, both
depths**, exactly like the NISAR notebook (20cm reuses the 5cm run's
dry-anchor date).

**Real differences from `03a`, driven by the S1 pipeline itself:**
- **VV only** -- these bursts are single-pol VV.
- **One coherence CSV per (station, orbit), not per station** --
  `S1_coherence_closure_alaska/{station}_orbit{orbit}_S1_coherence.csv`
  already is one group's full pairwise matrix, so there's no `groupby` step.
- **C-band, not L-band** -- `invert_sm` is called with `freq_GHz=5.405`
  (Sentinel-1) instead of NISAR's `1.2575`, materially changing the forward
  model, not just a label.
- **`local_incidence_angle_deg` comes from `_lv_theta.tif`** (the HyP3
  product's look-vector elevation angle, `90 - degrees(lv_theta)`), not a
  recomputed DEM-based angle.

In [1]:
import sys
import os
from pathlib import Path

import pyproj
pyproj.datadir.set_data_dir(os.path.join(sys.prefix, "Library", "share", "proj"))


def require_globals(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            f"Missing {missing} -- run the earlier setup cells first "
            "(the kernel may have been restarted)."
        )


BASE = Path.cwd()
COH_CLOSURE_DIR = BASE / "S1_coherence_closure_alaska"
INSITU_DIR = BASE / "ismn_data" / "alaska" / "station_data"
SOILGRIDS_DIR = BASE / "soilgrids_data" / "alaska"

if str(BASE / "INSAR4SM-main") not in sys.path:
    sys.path.append(str(BASE / "INSAR4SM-main"))

import numpy as np
import pandas as pd
import rasterio
from insar4sm.coh_funcs import calc_SM_coherences
from insar4sm.inversion import invert_sm


C:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## 1. Station list

In [2]:
require_globals("BASE")

import json

stations_path = BASE / "ismn_data" / "alaska" / "stations.geojson"
with open(stations_path) as f:
    stations_geojson = json.load(f)
stations = {
    feat["properties"]["station_val"]: (
        feat["geometry"]["coordinates"][1],  # lat
        feat["geometry"]["coordinates"][0],  # lon
    )
    for feat in stations_geojson["features"]
}
print(f"Loaded {len(stations)} stations")


Loaded 20 stations


## 2. In-situ SM reader and a generic bracket-interpolator

Identical to `03a_estimate_sm_nisar.ipynb` -- brackets the nearest real
sensor readings and caps the gap, rather than assuming a fixed grid.

In [3]:
require_globals("INSITU_DIR")

MAX_GAP_HOURS = 6
DEPTH_MAIN = 0.05
DEPTH_SECONDARY = 0.20

def read_insitu(station, depth):
    path = INSITU_DIR / f"{station}_ts_{depth:.2f}.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    return df.iloc[:, 0].sort_index()  # single sm_<depth> column, whatever it's named


def interp_insitu_at(series, ts):
    before = series.loc[:ts]
    after = series.loc[ts:]
    if before.empty or after.empty:
        return np.nan
    t0, v0 = before.index[-1], before.iloc[-1]
    t1, v1 = after.index[0], after.iloc[0]
    if t0 == t1:
        return float(v0)
    gap_hours = (t1 - t0).total_seconds() / 3600.0
    if gap_hours > MAX_GAP_HOURS:
        return np.nan
    frac = (ts - t0).total_seconds() / (t1 - t0).total_seconds()
    return float(v0 + frac * (v1 - v0))

## 3. Soil texture: SoilGrids

Identical to `03a_estimate_sm_nisar.ipynb`'s Section 3 -- same manual
download step, directory, two-tile fallback, and depth-layer naming
convention. In-situ ISMN texture isn't used.

In [4]:
require_globals("SOILGRIDS_DIR", "stations")


def sample_soilgrids(band_stem, lon, lat):
    for suffix in ("", "_ii"):
        path = SOILGRIDS_DIR / f"{band_stem}{suffix}.tif"
        if not path.exists():
            continue
        with rasterio.open(path) as src:
            row, col = src.index(lon, lat)
            if not (0 <= row < src.height and 0 <= col < src.width):
                continue
            val = float(src.read(1)[row, col])
            if 0 < val <= 1000:  # plausible g/kg range; SoilGrids has no explicit nodata sentinel here
                return val / 10.0  # g/kg -> %
    return None


def porosity_from_texture(sand_pct, clay_pct):
    return 100 * (0.332 - 7.251e-4 * sand_pct + 0.1276 * np.log10(clay_pct))


TEXTURE_LAYERS = {DEPTH_MAIN: "0to5cm", DEPTH_SECONDARY: "15to30cm"}

texture_soilgrids = {}
for station, (lat, lon) in stations.items():
    sg_by_depth = {}
    for depth, layer in TEXTURE_LAYERS.items():
        sand_sg = sample_soilgrids(f"sand_{layer}", lon, lat)
        clay_sg = sample_soilgrids(f"clay_{layer}", lon, lat)
        if sand_sg is not None and clay_sg is not None:
            sg_by_depth[depth] = {"sand_pct": sand_sg, "clay_pct": clay_sg, "porosity_pct": porosity_from_texture(sand_sg, clay_sg)}
    if sg_by_depth:
        texture_soilgrids[station] = sg_by_depth

print(f"SoilGrids texture: {len(texture_soilgrids)}/{len(stations)} station(s)")

SoilGrids texture: 19/20 station(s)


## 4. Load the precomputed S1 coherence CSVs, build per-(station, orbit) matrices

Discovers stations/orbits from `S1_coherence_closure_alaska/`'s filenames.
Each file already **is** one (station, orbit) group's full pairwise matrix,
so unlike `03a` there's no `groupby` step -- `coherence_VV * exp(i *
phase_VV)` reconstructs the interferometric value directly per row, and
`local_incidence_angle_deg` is averaged into `inc_angle_group[key]`.

In [5]:
require_globals("COH_CLOSURE_DIR", "stations")

import re

FILENAME_RE = re.compile(r"^(?P<station>.+)_orbit(?P<orbit>\d+)_S1_coherence\.csv$")
COH_THRESHOLD = 0.3  # below this, a pair is excluded from the coherence-fit term (see 03a_estimate_sm_nisar.ipynb)

slc_dates_group = {}
inc_angle_group = {}
coh_full_ds_group = {}  # (station, orbit) -> (1, n, n)
ph_ds_group = {}        # (station, orbit) -> (1, n, n)

for coh_path in sorted(COH_CLOSURE_DIR.glob("*_S1_coherence.csv")):
    m = FILENAME_RE.match(coh_path.name)
    if not m:
        continue
    station, orbit = m.group("station"), m.group("orbit")
    if station not in stations:
        continue

    df = pd.read_csv(coh_path)
    if "phase_VV" not in df.columns or "local_incidence_angle_deg" not in df.columns:
        raise RuntimeError(
            f"{coh_path} has no 'phase_VV'/'local_incidence_angle_deg' column -- "
            "it was written before 02c_calc_s1_coherence_closure.ipynb's Section 5 was "
            "fixed to save phase and incidence angle alongside coherence. "
            "Re-run that notebook's Section 5 (purely local, no new HyP3 jobs "
            "needed as long as S1_burst_data_alaska/coh_phase_closure/ still "
            "has the extracted pair products) to regenerate "
            "S1_coherence_closure_alaska/ with the current code, then re-run "
            "this notebook."
        )
    df["date_1"] = pd.to_datetime(df["date_1"])
    df["date_2"] = pd.to_datetime(df["date_2"])

    dates = sorted(set(df["date_1"]) | set(df["date_2"]))
    n = len(dates)
    if n < 3:
        continue  # can't form a phase-closure triplet
    date_idx = {d: i for i, d in enumerate(dates)}

    coh_full_ds = np.full((1, n, n), np.nan, dtype=float)
    ph_ds = np.zeros((1, n, n), dtype=np.complex128)
    for i in range(n):
        ph_ds[0, i, i] = 1.0 + 0j

    for row in df.itertuples():
        i, j = date_idx[row.date_1], date_idx[row.date_2]
        coh_val = row.coherence_VV
        phase_val = row.phase_VV
        if np.isnan(coh_val) or np.isnan(phase_val):
            continue
        ph_val = coh_val * np.exp(1j * phase_val)
        ph_ds[0, i, j] = ph_val
        ph_ds[0, j, i] = np.conj(ph_val)
        if coh_val < COH_THRESHOLD:
            coh_full_ds[0, i, j] = np.nan
            coh_full_ds[0, j, i] = np.nan
        else:
            coh_full_ds[0, i, j] = coh_val
            coh_full_ds[0, j, i] = coh_val

    key = (station, orbit)
    slc_dates_group[key] = dates
    inc_angle_group[key] = float(df["local_incidence_angle_deg"].mean())
    coh_full_ds_group[key] = coh_full_ds
    ph_ds_group[key] = ph_ds

print(f"{len(slc_dates_group)} (station, orbit) group(s) with >= 3 acquisitions")
for key, dates in slc_dates_group.items():
    print(f"  {key[0]} orbit {key[1]}: {len(dates)} acquisition(s)")


22 (station, orbit) group(s) with >= 3 acquisitions
  AtigunPass orbit 94: 6 acquisition(s)
  Coldfoot orbit 94: 6 acquisition(s)
  EagleSummit orbit 123: 6 acquisition(s)
  EagleSummit orbit 160: 4 acquisition(s)
  EagleSummit orbit 94: 6 acquisition(s)
  GobblersKnob orbit 94: 6 acquisition(s)
  GraniteCrk orbit 160: 4 acquisition(s)
  GraniteCrk orbit 94: 6 acquisition(s)
  ImnaviatCreek orbit 94: 6 acquisition(s)
  Kenai-29-ENE orbit 131: 6 acquisition(s)
  LittleChenaRidge orbit 160: 4 acquisition(s)
  LittleChenaRidge orbit 94: 6 acquisition(s)
  McGrath orbit 36: 4 acquisition(s)
  NukaGlacier orbit 131: 4 acquisition(s)
  PilgrimHotSprings orbit 153: 5 acquisition(s)
  RockyPoint orbit 153: 5 acquisition(s)
  SpringCreek orbit 131: 5 acquisition(s)
  SpringCreek orbit 160: 4 acquisition(s)
  SusitnaValleyHigh orbit 131: 5 acquisition(s)
  TurnagainPass orbit 160: 4 acquisition(s)
  UpperNomeCreek orbit 160: 4 acquisition(s)
  UpperNomeCreek orbit 94: 6 acquisition(s)


## 5. Per-(station, orbit) dry-date selection (5cm-anchored)

Identical logic to `03a_estimate_sm_nisar.ipynb` -- lowest interpolated 5cm
in-situ SM reading becomes the dry anchor, reused for both depth variants.

In [6]:
require_globals("slc_dates_group")

dry_idx_group = {}
dry_acq_time_group = {}
dry_bands_inds_group = {}
SM_sorting_group = {}
SM_index_init_group = {}
sm_dry_state_5cm_group = {}

for key, dates in slc_dates_group.items():
    station, orbit = key
    n = len(dates)
    series = read_insitu(station, DEPTH_MAIN)
    vals = None
    if series is not None:
        vals = np.array([interp_insitu_at(series, ts) for ts in dates])
    if vals is None or np.all(np.isnan(vals)):
        idx = 0
        sorting = np.arange(n)
        dry_val_pct = 14.0
    else:
        idx = int(np.nanargmin(vals))
        sorting = np.argsort(vals)  # NaN entries sort last, numpy's default
        dry_val_pct = float(vals[idx] * 100)
    dry_idx_group[key] = idx
    dry_acq_time_group[key] = dates[idx].to_pydatetime()
    dry_bands_inds_group[key] = np.array([idx], dtype=int)
    SM_sorting_group[key] = sorting
    init = np.empty(n)
    init[sorting] = np.linspace(10, 30, n)
    SM_index_init_group[key] = init
    sm_dry_state_5cm_group[key] = dry_val_pct

print("Per-(station, orbit) dry-anchor acquisition and 5cm dry state:")
for key in slc_dates_group:
    print(f"  {key[0]} orbit {key[1]}: {dry_acq_time_group[key]}  {sm_dry_state_5cm_group[key]:.2f}%")


Per-(station, orbit) dry-anchor acquisition and 5cm dry state:
  AtigunPass orbit 94: 2026-07-06 00:00:00  10.70%
  Coldfoot orbit 94: 2026-08-11 00:00:00  19.00%
  EagleSummit orbit 123: 2026-08-13 00:00:00  10.90%
  EagleSummit orbit 160: 2026-08-09 00:00:00  14.40%
  EagleSummit orbit 94: 2026-08-11 00:00:00  13.00%
  GobblersKnob orbit 94: 2026-07-06 00:00:00  12.80%
  GraniteCrk orbit 160: 2026-08-21 00:00:00  4.60%
  GraniteCrk orbit 94: 2026-08-23 00:00:00  4.30%
  ImnaviatCreek orbit 94: 2026-07-18 00:00:00  29.50%
  Kenai-29-ENE orbit 131: 2026-07-02 00:00:00  12.60%
  LittleChenaRidge orbit 160: 2026-08-09 00:00:00  9.70%
  LittleChenaRidge orbit 94: 2026-08-11 00:00:00  9.00%
  McGrath orbit 36: 2026-08-19 00:00:00  6.60%
  NukaGlacier orbit 131: 2026-08-19 00:00:00  23.70%
  PilgrimHotSprings orbit 153: 2026-07-10 00:00:00  27.80%
  RockyPoint orbit 153: 2026-07-10 00:00:00  34.30%
  SpringCreek orbit 131: 2026-07-02 00:00:00  23.40%
  SpringCreek orbit 160: 2026-08-09 00:0

## 6. 20cm dry state (same anchor date, different in-situ depth read)

In [7]:
require_globals("dry_acq_time_group")

sm_dry_state_20cm_group = {}
for key in slc_dates_group:
    station, orbit = key
    series = read_insitu(station, DEPTH_SECONDARY)
    if series is not None:
        val = interp_insitu_at(series, dry_acq_time_group[key])
        if np.isnan(val):
            val = series.min()
        sm_dry_state_20cm_group[key] = float(val * 100)
    else:
        sm_dry_state_20cm_group[key] = 14.0


## 7. Run the inversion

`run_inversion` is the shared body for both variants -- same call `03a`
uses, except `freq_GHz=5.405` (C-band, vs. NISAR's `1.2575`) and no `pol`
argument (VV only).

In [8]:
require_globals(
    "coh_full_ds_group", "ph_ds_group", "slc_dates_group", "inc_angle_group",
    "SM_sorting_group", "SM_index_init_group", "dry_bands_inds_group", "dry_acq_time_group",
)


def run_inversion(key, sm_dry_state, clay_pct, sand_pct, porosity_pct):
    n = len(slc_dates_group[key])
    slc_dates_arr = np.array([d.to_pydatetime() for d in slc_dates_group[key]], dtype=object)

    coh_sm, _, _ = calc_SM_coherences(
        Datetimes=slc_dates_arr,
        SM_sorting=SM_sorting_group[key],
        n_bands=n,
        n_ds=1,
        ind_DS=0,
        coh_full_DS=coh_full_ds_group[key],
        band_start=0,
        band_end=n,
        n_dry_bands=1,
        max_coh_flag=True,
        simple_exp_model=False,
        sm_only_mode=True,
    )

    sm_pix = invert_sm(
        ph_DS=ph_ds_group[key],
        coh_sm=coh_sm[0],
        SM_index=SM_index_init_group[key],
        dry_bands_inds=dry_bands_inds_group[key],
        slc_dates=slc_dates_arr,
        DS_ind=0,
        driest_date=dry_acq_time_group[key],
        inc_DS=np.array([inc_angle_group[key]]),
        band_start=0,
        band_end=n,
        nbands=n,
        opt_method="SLSQP",
        opt_parms={"ftol": 1e-3, "eps": 0.01, "maxiter": 50},
        ph_closure_dist=5,
        weight_factor=1.0,
        sm_dry_state=sm_dry_state,
        freq_GHz=5.405,  # Sentinel-1 C-band
        clay_pct=clay_pct,
        sand_pct=sand_pct,
    )
    return np.clip(sm_pix, 0, porosity_pct)


sm_map = {}  # (station, orbit, texture_source, depth_label) -> sm array
for key in slc_dates_group:
    station, orbit = key

    if station in texture_soilgrids:
        for depth, dry_state_group, depth_label in (
            (DEPTH_MAIN, sm_dry_state_5cm_group, "5cm"),
            (DEPTH_SECONDARY, sm_dry_state_20cm_group, "20cm"),
        ):
            tex = texture_soilgrids[station].get(depth)
            if tex is None:
                continue
            sm_pix = run_inversion(key, dry_state_group[key], tex["clay_pct"], tex["sand_pct"], tex["porosity_pct"])
            sm_map[(station, orbit, "soilgrids", depth_label)] = sm_pix
            print(f"  {station} orbit {orbit} [soilgrids, {depth_label}]: SM = {sm_pix.round(2)} %")

print(f"\nInversion complete: {len(sm_map)} (station, orbit, texture, depth) result(s)")

  AtigunPass orbit 94 [soilgrids, 5cm]: SM = [42.71 10.7  16.24  9.9  15.98 13.96] %


  AtigunPass orbit 94 [soilgrids, 20cm]: SM = [41.87 15.9  23.32 14.45 22.88 19.85] %


  Coldfoot orbit 94 [soilgrids, 5cm]: SM = [18.2  25.11 35.92 12.38 19.   10.82] %


  Coldfoot orbit 94 [soilgrids, 20cm]: SM = [19.29 34.02 41.19  8.8  21.6  24.79] %


  EagleSummit orbit 123 [soilgrids, 5cm]: SM = [15.44  9.07 10.55  9.07 10.9   9.93] %


...

  SusitnaValleyHigh orbit 131 [soilgrids, 5cm]: SM = [14.1  32.34 18.72 42.37 42.37] %


  SusitnaValleyHigh orbit 131 [soilgrids, 20cm]: SM = [33.8   4.06  0.   25.95 40.21] %
  TurnagainPass orbit 160 [soilgrids, 5cm]: SM = [14.   18.27 27.85 24.12] %
  TurnagainPass orbit 160 [soilgrids, 20cm]: SM = [14.   18.47 27.83 24.09] %


  UpperNomeCreek orbit 160 [soilgrids, 5cm]: SM = [18.82 27.3  26.5  30.67] %
  UpperNomeCreek orbit 160 [soilgrids, 20cm]: SM = [16.19 23.06 22.4  25.94] %


  UpperNomeCreek orbit 94 [soilgrids, 5cm]: SM = [10.55 34.36  3.98 22.03 26.2  24.4 ] %


  UpperNomeCreek orbit 94 [soilgrids, 20cm]: SM = [42.37 24.17 32.01 17.13 21.9  19.64] %

Inversion complete: 44 (station, orbit, texture, depth) result(s)


## 8. Export SM estimates

Writes `sm_map` to a CSV (station, orbit, texture_source, depth, date,
sm_pct, is_dry_anchor) so other notebooks can use these estimates without
re-running the inversion. `is_dry_anchor` flags the acquisition forced to
match the in-situ dry state by the inversion's own equality constraint.

In [9]:
require_globals("sm_map", "slc_dates_group", "dry_idx_group")

export_rows = []
for (station, orbit, texture_source, depth_label), sm_pix in sm_map.items():
    dates = slc_dates_group[(station, orbit)]
    dry_idx = dry_idx_group[(station, orbit)]
    for i, (date, sm_pct) in enumerate(zip(dates, sm_pix)):
        export_rows.append({
            "station": station, "orbit": orbit, "texture_source": texture_source,
            "depth": depth_label, "date": date, "sm_pct": sm_pct,
            "is_dry_anchor": i == dry_idx,
        })

sm_estimates_df = pd.DataFrame(export_rows).sort_values(["station", "orbit", "depth", "date"])
sm_estimates_path = BASE / "S1_SM_estimates_alaska.csv"
sm_estimates_df.to_csv(sm_estimates_path, index=False)
print(f"Wrote {len(sm_estimates_df)} row(s) to {sm_estimates_path}")

Wrote 224 row(s) to D:\Docs\RS_2023\NISAR4SM\S1_SM_estimates_alaska.csv


<hr>

## Summary

`sm_map` holds the per-(station, orbit, texture_source, depth) SM estimate
array (VV only). Exported above (Section 8) for the station-overview
notebook (`04`), which does the in-situ/SME2 comparison, screening, and
visualization. Re-run after `S1_coherence_closure_alaska/` gains more
completed HyP3 pairs, or SoilGrids/SME2 data changes -- everything here is
purely local, no network cost.

<hr>